# REST design

**Objective:** Design predictable resource operations using standard HTTP semantics.

## Simple version

In [ ]:
# REST routes name resources; HTTP methods express operations on them.
routes = {
    "GET /tasks": "list tasks",
    "POST /tasks": "create a task",
    "GET /tasks/1": "retrieve task 1",
    "PATCH /tasks/1": "update task 1",
    "DELETE /tasks/1": "delete task 1",
}

print(routes)

## Polished version

In [ ]:
# Separate create, update, and response schemas because their fields differ.
from typing import Optional

import httpx
from fastapi import FastAPI, HTTPException, Response, status
from pydantic import BaseModel, Field


class TaskCreate(BaseModel):
    title: str = Field(min_length=1, max_length=200)


class TaskUpdate(BaseModel):
    title: Optional[str] = Field(default=None, min_length=1, max_length=200)
    completed: Optional[bool] = None


class TaskResponse(BaseModel):
    id: int
    title: str
    completed: bool = False


def create_app() -> FastAPI:
    app = FastAPI()
    tasks: dict[int, TaskResponse] = {}

    @app.get("/tasks", response_model=list[TaskResponse])
    async def list_tasks() -> list[TaskResponse]:
        return list(tasks.values())

    @app.post("/tasks", response_model=TaskResponse, status_code=status.HTTP_201_CREATED)
    async def create_task(body: TaskCreate) -> TaskResponse:
        task = TaskResponse(id=len(tasks) + 1, title=body.title)
        tasks[task.id] = task
        return task

    @app.patch("/tasks/{task_id}", response_model=TaskResponse)
    async def update_task(task_id: int, body: TaskUpdate) -> TaskResponse:
        task = tasks.get(task_id)
        if task is None:
            raise HTTPException(404, "Task not found")
        # Change only fields the client actually sent.
        updated = task.model_copy(update=body.model_dump(exclude_unset=True))
        tasks[task_id] = updated
        return updated

    @app.delete("/tasks/{task_id}", status_code=status.HTTP_204_NO_CONTENT)
    async def delete_task(task_id: int) -> Response:
        if tasks.pop(task_id, None) is None:
            raise HTTPException(404, "Task not found")
        return Response(status_code=status.HTTP_204_NO_CONTENT)

    return app


# Test one resource through create, update, and delete operations.
app = create_app()
transport = httpx.ASGITransport(app=app)
async with httpx.AsyncClient(transport=transport, base_url="http://test") as client:
    created = await client.post("/tasks", json={"title": "Design resources"})
    task_id = created.json()["id"]
    updated = await client.patch(f"/tasks/{task_id}", json={"completed": True})
    deleted = await client.delete(f"/tasks/{task_id}")

print(created.status_code, updated.json(), deleted.status_code)